# Lab 5 — Notebook 02: Detection, Confidence, and Evaluation

**Core duration: 30 minutes.** Use `data/sample.jpg` and `yolo11n.pt`.

Learn what a pretrained detector returns, distinguish confidence from accuracy, calculate box overlap (IoU), and interpret detection metrics.

**Route:** load/infer 8 min → interpret/thresholds 8 min → IoU/evaluation 14 min. Extra class-counting and pose exploration are optional. Run the environment check first. Missing resources produce explicit skipped-inference messages; they do not count as completed exercises.


## 1. Load an image and a pretrained detector (8 min)

Classification assigns a label to an image. Detection predicts object classes **and locations**, potentially for several objects. A pretrained detector recognizes its trained vocabulary; it cannot recognize a new project class just because you rename a label.

The coffee photo is from scikit-image's sample data. Its predictions are not ground truth. Keep the same model version throughout the course.


In [ ]:
from pathlib import Path
# Support notebooks started in lab05/ or in the repository root.
LAB_DIR = Path.cwd().resolve()
if (LAB_DIR / "lab05" / "data" / "digits.csv").is_file():
    LAB_DIR = LAB_DIR / "lab05"
if not (LAB_DIR / "data" / "digits.csv").is_file():
    raise FileNotFoundError("Run this notebook in lab05/ or the repository root; keep data/ and scripts/ in place.")

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
image_path = LAB_DIR / "data/sample.jpg"
image = Image.open(image_path).convert("RGB")
plt.figure(figsize=(7, 4))
plt.imshow(image)
plt.axis("off")
plt.title("Input image — inspect it before seeing predictions")
plt.show()


### Load weights and infer

Use local weights found in `lab05/weights/`, `lab05/`, or the repository root. Set `ALLOW_DOWNLOAD=True` below only when weights should be downloaded. Inference defaults to CPU. Download/setup time is outside the 30-minute exercise budget.


In [ ]:
# Set this once before class. Download only when explicitly enabled.
ALLOW_DOWNLOAD = False
weight_candidates = [LAB_DIR / "weights/yolo11n.pt", LAB_DIR / "yolo11n.pt", LAB_DIR.parent / "yolo11n.pt"]
WEIGHTS = next((p for p in weight_candidates if p.is_file()), weight_candidates[0])
MODEL_SOURCE = str(WEIGHTS) if WEIGHTS.is_file() else ("yolo11n.pt" if ALLOW_DOWNLOAD else None)

model = None
result = None
try:
    from ultralytics import YOLO
except ImportError as exc:
    print("INFERENCE SKIPPED — install Lab 5 dependencies:", exc)
else:
    if MODEL_SOURCE is None:
        print("INFERENCE SKIPPED — provide yolo11n.pt or explicitly enable download.")
    else:
        model = YOLO(MODEL_SOURCE)
        result = model.predict(str(image_path), conf=0.25, imgsz=416, device="cpu", verbose=False)[0]
        print("Detections:", len(result.boxes))
        plt.figure(figsize=(8, 5))
        plt.imshow(result.plot()[..., ::-1])  # BGR to RGB
        plt.axis("off")
        plt.title("Predictions — not ground truth")
        plt.show()


## 2. Interpret predictions and compare thresholds (8 min)

`xyxy` contains pixel coordinates `(x_min, y_min, x_max, y_max)`. The class ID maps through `result.names`; IDs depend on the model's class vocabulary.

Confidence is a score attached to a prediction, **not overall model accuracy**, and should not automatically be read as a calibrated probability. A confident prediction can be wrong.


In [ ]:
if result is None:
    print("Prediction inspection skipped: first complete inference above.")
elif len(result.boxes)==0:
    print('No detections at this threshold. Try a lower confidence or a different photo.')
else:
    for box in result.boxes:
        cls=int(box.cls.item())
        name=result.names[cls]
        conf=float(box.conf.item())
        x1,y1,x2,y2=box.xyxy[0].tolist()
        print(f'{name:15} confidence={conf:.2f}  box=({x1:.0f},{y1:.0f},{x2:.0f},{y2:.0f})')


### Practice — Count detections (optional, 3 min)

If inference ran, count boxes by class. If there are none, report zero; do not assume every image must contain a detectable pretrained class.


In [ ]:
# TODO: Create a dictionary count_by_class.
# TODO: Loop over result.boxes and increment each count.
# TODO: Display the resulting dictionary.

### Core — Compare two confidence thresholds

Changing the confidence threshold filters predictions; it does **not** retrain the model. Lower thresholds often retain more detections but can also retain false positives. Choose deployment thresholds using validation observations, not final test data.


In [ ]:
if model is None:
    print("Threshold comparison skipped: first load the detector.")
else:
    thresholds = [0.15, 0.60]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, threshold in zip(axes, thresholds):
        r = model.predict(str(image_path), conf=threshold, imgsz=416, device="cpu", verbose=False)[0]
        ax.imshow(r.plot()[..., ::-1])
        ax.set_title(f"conf ≥ {threshold:.2f} | {len(r.boxes)} boxes")
        ax.axis("off")
    plt.tight_layout()
    plt.show()


### Core reflection

Name one retained, removed, or missed object. Would missing an object or falsely detecting one be more costly in your proposed application? Explain why the displayed boxes alone cannot establish precision or recall: independent annotations are needed.

**Your observations:** TODO.


In [ ]:
# OPTIONAL: try a third threshold after completing the core IoU exercise below


## 3. When is a box correct? IoU and matching (14 min)

**Intersection over Union:**

$$\mathrm{IoU} = \frac{\text{area shared by both boxes}}{\text{area covered by either box}}.$$

An identical pair has IoU 1; disjoint boxes have IoU 0. IoU measures localization, not class correctness. Use continuous pixel coordinates here (no `+1` in width/height).

In this simplified exercise, a detection is a true positive if its class is correct and IoU ≥ 0.50 with an **unmatched** ground-truth object. Each ground-truth object can match only one prediction. Evaluate predictions in descending confidence order; extra boxes for an already matched object are false positives. Unmatched ground-truth objects are false negatives.

The example below is invented geometry, not a measured evaluation of the coffee detector.


In [ ]:
import matplotlib.patches as patches

def box_iou(a, b):
    """IoU for two valid xyxy boxes in continuous image coordinates."""
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    if a.shape != (4,) or b.shape != (4,) or not np.isfinite([a, b]).all():
        raise ValueError("Expected two finite xyxy boxes.")
    if a[2] <= a[0] or a[3] <= a[1] or b[2] <= b[0] or b[3] <= b[1]:
        raise ValueError("Boxes need positive width and height.")
    intersection = max(0., min(a[2], b[2]) - max(a[0], b[0])) * max(0., min(a[3], b[3]) - max(a[1], b[1]))
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    return float(intersection / (area_a + area_b - intersection))

ground_truth = [20, 20, 100, 100]
predicted = [40, 20, 120, 100]
print(f"IoU: {box_iou(ground_truth, predicted):.2f}")
fig, ax = plt.subplots(figsize=(5, 4))
for box, color, label in [(ground_truth, "green", "Ground truth: cup"), (predicted, "blue", "Prediction: cup")]:
    x1, y1, x2, y2 = box
    ax.add_patch(patches.Rectangle((x1, y1), x2-x1, y2-y1, fill=False, edgecolor=color, linewidth=2, label=label))
ax.set(xlim=(0, 150), ylim=(140, 0), xlabel="x (pixels)", ylabel="y (pixels)", title="Overlap of two boxes")
ax.set_aspect("equal")
ax.legend()
plt.tight_layout()
plt.show()


### Core exercise — Match predictions by hand

First verify the printed IoU using the shared area and union area. Then consider **one** ground-truth cup and these predictions in descending confidence order:

| Prediction | Confidence | Predicted class | IoU with the cup |
|---|---:|---|---:|
| A | 0.90 | cup | 0.60 |
| B | 0.80 | cup | 0.80 |
| C | 0.70 | bottle | 0.90 |

At confidence ≥ 0.50 and matching IoU ≥ 0.50, count TP, FP, and FN. Why does B not become a second TP? Why is C wrong despite high overlap? Calculate precision and recall.

Then raise the matching IoU threshold to 0.75 and repeat the matching. This changes the **evaluation criterion**, not the confidence threshold.

**Your answer:** TODO. Optional code: use `box_iou` to construct one disjoint pair and one identical pair.


In [ ]:
# TODO: work through TP/FP/FN and precision/recall in the Markdown response
# OPTIONAL: use box_iou with another valid pair


### From one image to a dataset

- **Precision:** TP / (TP + FP): how many retained detections are correct?
- **Recall:** TP / (TP + FN): how many labeled objects are found?
- **AP:** summarizes a class's precision–recall curve as the confidence cutoff varies, at a specified IoU matching threshold.
- **mAP50:** mean AP across classes, using matching IoU 0.50.
- **mAP50–95:** additionally averages across matching IoU thresholds 0.50, 0.55, …, 0.95, rewarding tighter localization.

Confidence filtering, IoU used for ground-truth matching, and IoU used by non-maximum suppression (duplicate filtering during prediction) are different settings. Do not interpret the `iou` prediction argument as the validation matching threshold. Exact evaluation implementations have additional matching details; use the library's validation routine for project metrics.

**Checkpoint:** Explain one false positive, one false negative, and why a confidence of 0.9 is not 90% dataset accuracy.

**Next:** Collect/inspect labels, train a small custom detector, validate, reload its weights, and run it through the webcam script.

**Optional pose exploration:** Pose adds ordered keypoints and different labels/metrics. It is outside the core detection route.

References: [Ultralytics prediction](https://docs.ultralytics.com/modes/predict/), [validation](https://docs.ultralytics.com/modes/val/).
